## 4. CRISP-DM: Data Preparation

In [ ]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from IPython.display import display
import re
import os
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, accuracy_score
from IPython.display import display, Markdown
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy import stats
 
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, confusion_matrix, recall_score, precision_score, roc_curve, f1_score

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Load dataset
df = pd.read_csv('../data/Cardiovascular_Disease_Dataset_1.csv')

# RuntimeError: Scikit-learn array API support was enabled but scipy's own support is not enabled. Please set the SCIPY_ARRAY_API=1 environment variable before importing sklearn or scipy. More details at: https://docs.scipy.org/doc/scipy/dev/api-dev/array_api.html
os.environ['SCIPY_ARRAY_API'] = '1'

### 4.1 Data Selection

In [ ]:
# Display basic information about the dataset
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst few rows:")
display(df.head())

#### Feature Analysis and Selection

In [ ]:
# Analyze data types and missing values
print("Data types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

# Check for duplicates
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Remove duplicates if any
df = df.drop_duplicates()

# Basic statistics
print("\nBasic statistics:")
display(df.describe())

#### Feature Engineering

In [ ]:
# Create age groups
df['age_group'] = pd.cut(df['age'], bins=[0, 40, 50, 60, 100], labels=['<40', '40-50', '50-60', '60+'])

# Create BMI proxy (not available directly, but we can create risk categories based on cholesterol)
df['cholesterol_risk'] = pd.cut(df['serumcholestrol'], 
                               bins=[0, 200, 240, np.inf], 
                               labels=['Normal', 'Borderline', 'High'])

# Create blood pressure categories
df['bp_category'] = pd.cut(df['restingBP'], 
                          bins=[0, 120, 140, np.inf], 
                          labels=['Normal', 'Elevated', 'High'])

# Create heart rate categories
df['hr_category'] = pd.cut(df['maxheartrate'], 
                          bins=[0, 100, 150, np.inf], 
                          labels=['Low', 'Normal', 'High'])

print("New engineered features created:")
print("- age_group")
print("- cholesterol_risk") 
print("- bp_category")
print("- hr_category")

### 4.2 Data Cleaning

In [ ]:
# Check for outliers using IQR method
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Check for outliers in numerical columns
numerical_cols = ['age', 'restingBP', 'serumcholestrol', 'maxheartrate', 'oldpeak']

for col in numerical_cols:
    outliers, lower, upper = detect_outliers_iqr(df, col)
    print(f"\n{col}:")
    print(f"  Outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)")
    print(f"  Range: {lower:.2f} - {upper:.2f}")

# Visualize outliers
plt.figure(figsize=(15, 8))
for i, col in enumerate(numerical_cols, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(y=df[col])
    plt.title(f'{col} - Outlier Detection')
plt.tight_layout()
plt.show()

# Handle outliers - cap extreme values
def cap_outliers(df, column, lower_percentile=0.01, upper_percentile=0.99):
    lower_cap = df[column].quantile(lower_percentile)
    upper_cap = df[column].quantile(upper_percentile)
    df[column] = np.clip(df[column], lower_cap, upper_cap)
    return df

# Apply outlier capping to extreme values only
for col in ['restingBP', 'serumcholestrol']:
    df = cap_outliers(df, col, 0.005, 0.995)
    print(f"Applied outlier capping to {col}")

##### Missing Values Handling

In [ ]:
# Handle missing values represented as zeros
print("Missing values analysis:")
print(f"Oldpeak zeros: {(df['oldpeak'] == 0).sum()} ({(df['oldpeak'] == 0).sum()/len(df)*100:.1f}%)")
print(f"Serum cholesterol zeros: {(df['serumcholestrol'] == 0).sum()} ({(df['serumcholestrol'] == 0).sum()/len(df)*100:.1f}%)")

# Handle serum cholesterol: 0 values are clearly missing data (all in heart disease patients)
# Replace with median of non-zero values
serum_chol_median = df[df['serumcholestrol'] != 0]['serumcholestrol'].median()
df['serumcholestrol'] = df['serumcholestrol'].replace(0, serum_chol_median)
print(f"Replaced serum cholesterol zero values with median of non-zero values: {serum_chol_median}")

# Handle oldpeak: Some zeros may be legitimate clinical values
# Use median imputation for zeros since they're a small percentage and mostly in healthy patients
oldpeak_median = df[df['oldpeak'] != 0]['oldpeak'].median()
df['oldpeak'] = df['oldpeak'].replace(0, oldpeak_median)
print(f"Replaced oldpeak zero values with median of non-zero values: {oldpeak_median}")

# Handle fasting blood sugar zeros if present
if 'fastingbloodsugar' in df.columns:
    fbs_zero_count = (df['fastingbloodsugar'] == 0).sum()
    if fbs_zero_count > 0:
        fbs_median = df[df['fastingbloodsugar'] != 0]['fastingbloodsugar'].median()
        df['fastingbloodsugar'] = df['fastingbloodsugar'].replace(0, fbs_median)
        print(f"Replaced fasting blood sugar zero values with median of non-zero values: {fbs_median}")

print("Missing values (represented as zeros) have been handled using median imputation.")

### 4.3 Data Transformation and Integration

In [ ]:
X = df.drop('target', axis=1)
y = df['target']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts()}")

# Handle categorical variables with proper encoding
categorical_features = ['age_group', 'cholesterol_risk', 'bp_category', 'hr_category']

# One-hot encode categorical features
X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=True)

print(f"\nFeatures after encoding: {X_encoded.shape}")
print("New columns after encoding:")
print([col for col in X_encoded.columns if col not in X.columns])

# Remove redundant features after engineering (keep engineered features, remove original ones that were engineered)
# Remove original features that were used to create engineered features
# Keep: age (used for age_group but still valuable), restingBP (used for bp_category but still valuable), 
# serumcholestrol (used for cholesterol_risk but still valuable), maxheartrate (used for hr_category but still valuable)
# All original features remain valuable alongside their engineered versions
print(f"\nUsing all features after engineering: {X_encoded.shape[1]} features")

# Scale numerical features
scaler = StandardScaler()
numerical_features = ['age', 'restingBP', 'serumcholestrol', 'maxheartrate', 'oldpeak']

X_scaled = X_encoded.copy()
X_scaled[numerical_features] = scaler.fit_transform(X_encoded[numerical_features])

print("\nNumerical features have been standardized.")

#### Feature Selection Based on Correlation and Importance

In [ ]:
# Calculate correlation with target BEFORE feature engineering
X_before_engineering = df.drop('target', axis=1).select_dtypes(include=[np.number])
correlations_before = X_before_engineering.corrwith(y).abs().sort_values(ascending=False)
print("Feature correlations with target BEFORE feature engineering (top 10):")
print(correlations_before.head(10))

# Heatmap showing correlations to target BEFORE feature engineering
plt.figure(figsize=(8, 10))
target_correlations_before = X_before_engineering.corrwith(y).sort_values(ascending=False)
plt.subplot(2, 1, 1)
sns.heatmap(target_correlations_before.values.reshape(-1, 1), 
            yticklabels=target_correlations_before.index, 
            xticklabels=['Target Correlation'],
            annot=True, cmap='coolwarm', center=0, fmt='.3f')
plt.title('Feature Correlations to Target - BEFORE Feature Engineering')
plt.tight_layout()

# Calculate correlation with target AFTER feature engineering
correlations_after = X_encoded.corrwith(y).abs().sort_values(ascending=False)
print("\nFeature correlations with target AFTER feature engineering (top 10):")
print(correlations_after.head(10))

# Heatmap showing correlations to target AFTER feature engineering
target_correlations_after = X_encoded.corrwith(y).sort_values(ascending=False)
plt.subplot(2, 1, 2)
sns.heatmap(target_correlations_after.values.reshape(-1, 1), 
            yticklabels=target_correlations_after.index, 
            xticklabels=['Target Correlation'],
            annot=True, cmap='coolwarm', center=0, fmt='.3f')
plt.title('Feature Correlations to Target - AFTER Feature Engineering')
plt.tight_layout()
plt.show()

# Feature importance using Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_encoded, y)

feature_importance = pd.DataFrame({
    'feature': X_encoded.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 most important features (Random Forest):")
print(feature_importance.head(10))

# Plot feature importance
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

#### Final Dataset Preparation

In [ ]:
# Use all features for modeling (no feature selection)
X_final = X_scaled

print(f"Final feature set shape: {X_final.shape}")
print("Using all features:")
for i, feature in enumerate(X_final.columns, 1):
    print(f"{i:2d}. {feature}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Training target distribution:\n{y_train.value_counts()}")
print(f"Test target distribution:\n{y_test.value_counts()}")

#### Handle Class Imbalance (if needed)

In [ ]:
# Check class balance
class_counts = y_train.value_counts()
class_ratio = class_counts.min() / class_counts.max()
print(f"Class ratio: {class_ratio:.3f}")

print("Using original class distribution without resampling.")
X_train_final = X_train
y_train_final = y_train

#### Save Prepared Data

In [ ]:
# Save the prepared datasets
# Create data directory if it doesn't exist
os.makedirs('../data', exist_ok=True)

# Save processed data
X_train_final.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train_final.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

# Save the full processed dataset
processed_df = pd.concat([X_final, y], axis=1)
processed_df.to_csv('../data/cardiovascular_processed.csv', index=False)

# Save feature names and scaler for future use
import joblib
joblib.dump(scaler, '../data/scaler.pkl')
joblib.dump(X_final.columns.tolist(), '../data/selected_features.pkl')

print("\nData preparation completed!")
print("Saved files:")
print("- X_train.csv: Training features")
print("- X_test.csv: Test features") 
print("- y_train.csv: Training targets")
print("- y_test.csv: Test targets")
print("- cardiovascular_processed.csv: Full processed dataset")
print("- scaler.pkl: Fitted StandardScaler")
print("- selected_features.pkl: List of all feature names")

### Data Preparation Summary

The data preparation phase included:

1. **Data Loading and Initial Analysis**: Loaded the cardiovascular disease dataset with 1000 records and 14 features.

2. **Feature Engineering**: Created new features including age groups, cholesterol risk categories, blood pressure categories, and heart rate categories.

3. **Data Cleaning**: 
   - Detected and handled outliers using IQR method and percentile capping
   - Checked for missing values (none found in this dataset)
   - Removed duplicate records

4. **Data Transformation**:
   - One-hot encoded categorical variables
   - Standardized numerical features using StandardScaler
   - Used all features (no feature selection to maintain full information)

5. **Class Balance**: Evaluated class distribution and used original distribution without resampling.

6. **Train-Test Split**: Split data into 80% training and 20% test sets with stratification.

7. **Data Export**: Saved processed datasets and preprocessing objects for modeling phase.

The final dataset is ready for machine learning model training and evaluation.